# 00 · Join UBIGEO y carga de fuentes de datos

Notebook de la Etapa 0 del pipeline: antes de tocar una sola imagen, se cargan y verifican todas las fuentes de datos y se construye la tabla maestra unida por UBIGEO.

## Setup

In [4]:
import pandas as pd
import geopandas as gpd
import requests

pd.set_option("display.max_columns", 50)


In [5]:
from pathlib import Path

def find_project_root(marker="requirements.txt"):
    path = Path.cwd()
    for parent in [path] + list(path.parents):
        if (parent / marker).exists():
            return parent
    raise FileNotFoundError(f"No se encontró '{marker}' subiendo desde {path}")

PROJECT_ROOT = find_project_root()
DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA = PROJECT_ROOT / "data"

print("Project root:", PROJECT_ROOT)

Project root: c:\Users\JHOSSEP\Documents\REPO\causal-anemia-model


# Polígonos y ubicación geográfica

## Límite distrital INEI 2025 (shapefile)

Fuente: https://www.geogpsperu.com/2019/05/limite-distrital-actualizado-inei.html

Descarga manual (no tiene link directo de descarga)

In [7]:
ruta_limite_distrital = DATA_RAW / "Limite Distrital INEI 2025 CPV" / "Limite Distrital INEI 2025 CPV.shp"

limite_distrital = gpd.read_file(ruta_limite_distrital)
print(limite_distrital.shape)
limite_distrital.head()


(1891, 10)


,UBIGEO,CCDD,CCPP,CCDI,DEPARTAMEN,PROVINCIA,DISTRITO,OBJECTID,ESRI_OID,geometry
0,010101,01,01,01,AMAZONAS,CHACHAPOYAS,CHACHAPOYAS,1.0,5.0,"POLYGON ((-77.8858 -6.1778, -77.88323 -6.17846..."
1,010102,01,01,02,AMAZONAS,CHACHAPOYAS,ASUNCION,2.0,6.0,"POLYGON ((-77.74482 -5.94497, -77.74482 -5.945..."
2,010103,01,01,03,AMAZONAS,CHACHAPOYAS,BALSAS,3.0,7.0,"POLYGON ((-77.9358 -6.69039, -77.93531 -6.6909..."
3,010104,01,01,04,AMAZONAS,CHACHAPOYAS,CHETO,4.0,8.0,"POLYGON ((-77.71486 -6.24598, -77.71485 -6.245..."
4,010105,01,01,05,AMAZONAS,CHACHAPOYAS,CHILIQUIN,5.0,9.0,"POLYGON ((-77.77405 -5.99598, -77.77328 -5.996..."


## Tabla de UBIGEO (crosswalk departamento-provincia-distrito)

Fuente (CSV público, se puede leer directo desde la URL):
https://raw.githubusercontent.com/jmcastagnetto/ubigeo-peru-aumentado/main/ubigeo_distrito.csv

In [8]:
url_ubigeo = "https://raw.githubusercontent.com/jmcastagnetto/ubigeo-peru-aumentado/main/ubigeo_distrito.csv"

ubigeo = pd.read_csv(url_ubigeo, dtype={"inei": str, "reniec": str})

print(ubigeo.shape)
ubigeo.head()

(1893, 20)


,inei,reniec,departamento,provincia,distrito,region,macroregion_inei,macroregion_minsa,iso_3166_2,fips,capital,superficie,pob_densidad_2020,altitude,latitude,longitude,indice_vulnerabilidad_alimentaria,idh_2019,pct_pobreza_total,pct_pobreza_extrema
0,010101,010101,AMAZONAS,CHACHAPOYAS,CHACHAPOYAS,AMAZONAS,ORIENTE,MACROREGION ORIENTE,PE-AMA,1,Chachapoyas,153.78,201.43711796072299,2338.0,-6.229444,-77.872778,0.279657,0.642361,9.034625,1.439875
1,010102,010102,AMAZONAS,CHACHAPOYAS,ASUNCION,AMAZONAS,ORIENTE,MACROREGION ORIENTE,PE-AMA,1,Asunción,25.71,13.9634383508363,2823.0,-6.032500,-77.710833,0.558549,0.423032,36.519949,15.680750
2,010103,010103,AMAZONAS,CHACHAPOYAS,BALSAS,AMAZONAS,ORIENTE,MACROREGION ORIENTE,PE-AMA,1,Balsas,357.09,4.0465988966367004,859.0,-6.835833,-78.019722,0.646749,0.315308,45.732962,15.427120
3,010104,010104,AMAZONAS,CHACHAPOYAS,CHETO,AMAZONAS,ORIENTE,MACROREGION ORIENTE,PE-AMA,1,Cheto,56.97,13.568544848165701,2143.0,-6.255556,-77.700833,0.530846,0.345746,39.169782,23.678410
4,010105,010105,AMAZONAS,CHACHAPOYAS,CHILIQUIN,AMAZONAS,ORIENTE,MACROREGION ORIENTE,PE-AMA,1,Chiliquín,143.43,6.8326012689116604,2677.0,-6.078333,-77.737500,0.706854,0.275038,53.045662,36.395370


Como tenemos diferente cantidad de filas, verificaremos cuáles sonla que faltan en la 1ra

In [9]:
fila_nan = ubigeo[ubigeo["inei"].isna()]
fila_nan

,inei,reniec,departamento,provincia,distrito,region,macroregion_inei,macroregion_minsa,iso_3166_2,fips,capital,superficie,pob_densidad_2020,altitude,latitude,longitude,indice_vulnerabilidad_alimentaria,idh_2019,pct_pobreza_total,pct_pobreza_extrema
1892,NaN,170107,MOQUEGUA,MARISCAL NIETO,SAN ANTONIO,MOQUEGUA,SUR,MACROREGION SUR,PE-MOQ,18,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [10]:
# Quitar la fila con inei vacío (San Antonio, Moquegua - dato incompleto)
ubigeo = ubigeo[ubigeo["inei"].notna()].copy()
print("Distritos después de quitar el nan:", ubigeo.shape[0])

Distritos después de quitar el nan: 1892


In [11]:
ubigeo[ubigeo["inei"].isna()]

,inei,reniec,departamento,provincia,distrito,region,macroregion_inei,macroregion_minsa,iso_3166_2,fips,capital,superficie,pob_densidad_2020,altitude,latitude,longitude,indice_vulnerabilidad_alimentaria,idh_2019,pct_pobreza_total,pct_pobreza_extrema


In [12]:
# Los que están en el shapefile pero no en la tabla UBIGEO
print("--- En shapefile, no en tabla ---")
print(limite_distrital[limite_distrital["UBIGEO"].isin(["180107", "130112"])][["UBIGEO", "DEPARTAMEN", "PROVINCIA", "DISTRITO"]])

print()

# Los que están en la tabla UBIGEO pero no en el shapefile
print("--- En tabla, no en shapefile ---")
print(ubigeo[ubigeo["inei"].isin(["160109", "150144", "160114"])][["inei", "departamento", "provincia", "distrito"]])

--- En shapefile, no en tabla ---
      UBIGEO   DEPARTAMEN       PROVINCIA       DISTRITO
1182  130112  LA LIBERTAD        TRUJILLO  ALTO TRUJILLO
1534  180107     MOQUEGUA  MARISCAL NIETO    SAN ANTONIO

--- En tabla, no en shapefile ---
        inei departamento provincia                 distrito
1335  150144         LIMA      LIMA  SANTA MARIA DE HUACHIPA
1472  160109       LORETO    MAYNAS                 PUTUMAYO
1476  160114       LORETO    MAYNAS  TENIENTE MANUEL CLAVERO


In [13]:
ubigeos_finales = set(limite_distrital["UBIGEO"].astype(str)) & set(ubigeo["inei"].astype(str))
print("Distritos finales para el análisis:", len(ubigeos_finales))

limite_distrital = limite_distrital[limite_distrital["UBIGEO"].isin(ubigeos_finales)].copy()
ubigeo = ubigeo[ubigeo["inei"].isin(ubigeos_finales)].copy()

print("Shapefile final:", limite_distrital.shape[0])
print("Tabla ubigeo final:", ubigeo.shape[0])

Distritos finales para el análisis: 1889
Shapefile final: 1889
Tabla ubigeo final: 1889


# Megear y descargar

In [14]:
# Asegurar que el código UBIGEO tenga el mismo formato en ambas tablas (string, 6 dígitos)
limite_distrital["UBIGEO"] = limite_distrital["UBIGEO"].astype(str).str.zfill(6)
ubigeo["inei"] = ubigeo["inei"].astype(str).str.zfill(6)

# Del shapefile solo nos interesa el UBIGEO y la geometría
# (los demás campos administrativos -departamento, provincia, distrito- ya están más completos en la tabla ubigeo)
limite_distrital_geo = limite_distrital[["UBIGEO", "geometry"]].copy()

# Merge: atributos de la tabla ubigeo + geometría del shapefile
geobase = ubigeo.merge(
    limite_distrital_geo,
    left_on="inei",
    right_on="UBIGEO",
    how="inner" # todos los ubigeos coincidentes (ya filtramos los que no coincidían)
)

# Reconvertir a GeoDataFrame (el merge con un DataFrame normal devuelve un DataFrame plano)
geobase = gpd.GeoDataFrame(geobase, geometry="geometry", crs=limite_distrital.crs)

print("Geobase final:", geobase.shape)
geobase.head()

Geobase final: (1889, 22)


,inei,reniec,departamento,provincia,distrito,region,macroregion_inei,macroregion_minsa,iso_3166_2,fips,capital,superficie,pob_densidad_2020,altitude,latitude,longitude,indice_vulnerabilidad_alimentaria,idh_2019,pct_pobreza_total,pct_pobreza_extrema,UBIGEO,geometry
0,010101,010101,AMAZONAS,CHACHAPOYAS,CHACHAPOYAS,AMAZONAS,ORIENTE,MACROREGION ORIENTE,PE-AMA,1,Chachapoyas,153.78,201.43711796072299,2338.0,-6.229444,-77.872778,0.279657,0.642361,9.034625,1.439875,010101,"POLYGON ((-77.8858 -6.1778, -77.88323 -6.17846..."
1,010102,010102,AMAZONAS,CHACHAPOYAS,ASUNCION,AMAZONAS,ORIENTE,MACROREGION ORIENTE,PE-AMA,1,Asunción,25.71,13.9634383508363,2823.0,-6.032500,-77.710833,0.558549,0.423032,36.519949,15.680750,010102,"POLYGON ((-77.74482 -5.94497, -77.74482 -5.945..."
2,010103,010103,AMAZONAS,CHACHAPOYAS,BALSAS,AMAZONAS,ORIENTE,MACROREGION ORIENTE,PE-AMA,1,Balsas,357.09,4.0465988966367004,859.0,-6.835833,-78.019722,0.646749,0.315308,45.732962,15.427120,010103,"POLYGON ((-77.9358 -6.69039, -77.93531 -6.6909..."
3,010104,010104,AMAZONAS,CHACHAPOYAS,CHETO,AMAZONAS,ORIENTE,MACROREGION ORIENTE,PE-AMA,1,Cheto,56.97,13.568544848165701,2143.0,-6.255556,-77.700833,0.530846,0.345746,39.169782,23.678410,010104,"POLYGON ((-77.71486 -6.24598, -77.71485 -6.245..."
4,010105,010105,AMAZONAS,CHACHAPOYAS,CHILIQUIN,AMAZONAS,ORIENTE,MACROREGION ORIENTE,PE-AMA,1,Chiliquín,143.43,6.8326012689116604,2677.0,-6.078333,-77.737500,0.706854,0.275038,53.045662,36.395370,010105,"POLYGON ((-77.77405 -5.99598, -77.77328 -5.996..."


In [16]:
# Columnas de la tabla ubigeo que sí llevamos a la geobase
# (se excluyen reniec -duplica inei-, iso_3166_2 y fips -códigos administrativos poco útiles-,
# y los índices socioeconómicos -idh, pobreza, vulnerabilidad- que se cargarán después en el modelo causal)
cols_geobase = [
    "inei", "departamento", "provincia", "distrito",
    "region", "macroregion_inei", "macroregion_minsa",
    "capital", "latitude", "longitude", "altitude",
    "superficie", "pob_densidad_2020"
]

geobase = ubigeo[cols_geobase].merge(
    limite_distrital_geo,
    left_on="inei",
    right_on="UBIGEO",
    how="inner"
).drop(columns=["UBIGEO"]).rename(columns={"inei": "UBIGEO"})

geobase = gpd.GeoDataFrame(geobase, geometry="geometry", crs=limite_distrital.crs)

print("Geobase final:", geobase.shape)
geobase.head()

Geobase final: (1889, 14)


,UBIGEO,departamento,provincia,distrito,region,macroregion_inei,macroregion_minsa,capital,latitude,longitude,altitude,superficie,pob_densidad_2020,geometry
0,010101,AMAZONAS,CHACHAPOYAS,CHACHAPOYAS,AMAZONAS,ORIENTE,MACROREGION ORIENTE,Chachapoyas,-6.229444,-77.872778,2338.0,153.78,201.43711796072299,"POLYGON ((-77.8858 -6.1778, -77.88323 -6.17846..."
1,010102,AMAZONAS,CHACHAPOYAS,ASUNCION,AMAZONAS,ORIENTE,MACROREGION ORIENTE,Asunción,-6.032500,-77.710833,2823.0,25.71,13.9634383508363,"POLYGON ((-77.74482 -5.94497, -77.74482 -5.945..."
2,010103,AMAZONAS,CHACHAPOYAS,BALSAS,AMAZONAS,ORIENTE,MACROREGION ORIENTE,Balsas,-6.835833,-78.019722,859.0,357.09,4.0465988966367004,"POLYGON ((-77.9358 -6.69039, -77.93531 -6.6909..."
3,010104,AMAZONAS,CHACHAPOYAS,CHETO,AMAZONAS,ORIENTE,MACROREGION ORIENTE,Cheto,-6.255556,-77.700833,2143.0,56.97,13.568544848165701,"POLYGON ((-77.71486 -6.24598, -77.71485 -6.245..."
4,010105,AMAZONAS,CHACHAPOYAS,CHILIQUIN,AMAZONAS,ORIENTE,MACROREGION ORIENTE,Chiliquín,-6.078333,-77.737500,2677.0,143.43,6.8326012689116604,"POLYGON ((-77.77405 -5.99598, -77.77328 -5.996..."


In [17]:
# Guardar la geobase distrital en data/clean/staging
output_path = DATA / "clean" / "staging" / "geobase_distrital.gpkg"
output_path.parent.mkdir(parents=True, exist_ok=True)

geobase.to_file(output_path, driver="GPKG")
print("Guardado en:", output_path)

Guardado en: c:\Users\JHOSSEP\Documents\REPO\causal-anemia-model\data\clean\staging\geobase_distrital.gpkg
